# Phase 2 — Collecte : le scraper citoyen
**Durée estimée : 1h15**

---

## Objectif
Extraire des données structurées (titre, date, url) depuis une source **autorisée** (Wikipedia ou data.gouv.fr),  
en respectant les **3 règles d'or** du scraping responsable.

## Les 3 règles d'or
| Règle | Pourquoi |
|-------|----------|
| **Identification** | User-Agent avec votre email → le site peut vous contacter plutôt que de vous bloquer |
| **Politesse** | `time.sleep(2)` entre chaque page → on ne sature pas le serveur |
| **Traçabilité** | Logging des statuts HTTP → vous pouvez prouver votre démarche |

## Livrables de cette phase
- Votre code de collecte (ce notebook)
- Un fichier `data/export.csv` (≥ 20 lignes, colonnes : `source`, `titre`, `date`, `url`)
- Une justification de vos choix (cellule Markdown en fin de notebook)

---
> ⏱️ **Règle des 45 min :** Si à 45 min vous n'avez pas un CSV propre avec ≥ 20 lignes,  
> **arrêtez et passez au dataset de secours** `data/dataset_secours.csv` → notebook `03_audit.ipynb`.

In [10]:
# --- Installation (décommentez si vous êtes sur Google Colab) ---
# !pip install requests beautifulsoup4 pandas lxml

In [11]:
import requests
import pandas as pd
import time
import logging
import os
from bs4 import BeautifulSoup
from datetime import datetime

# Création du dossier de sortie
os.makedirs("data", exist_ok=True)

print("Imports OK ✓")

Imports OK ✓


---
## Étape 1 — Configuration du scraper

**Modifiez la variable `VOTRE_EMAIL`** avec votre adresse email réelle.  
C'est votre signature numérique : elle prouve votre bonne foi si un admin web vous contacte.

In [12]:
# ✏️  MODIFIEZ avec votre email
VOTRE_EMAIL = "etudiant@ecole.fr"

# Configuration des headers HTTP — identification claire
HEADERS = {
    "User-Agent": f"TP-Scraping-Pedagogique/1.0 (contact: {VOTRE_EMAIL}; but: formation)"
}

# Pause entre les requêtes (en secondes) — règle de politesse
PAUSE_SECONDES = 2

# Configuration du logging pour tracer vos requêtes
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

print(f"User-Agent configuré : {HEADERS['User-Agent']}")

User-Agent configuré : TP-Scraping-Pedagogique/1.0 (contact: etudiant@ecole.fr; but: formation)


---
## Étape 2 — Fonction utilitaire : requête HTTP sécurisée

Cette fonction gère les cas d'erreur courants (timeout, 429 Too Many Requests, 503) et logue chaque requête.

In [13]:
def get_page(url: str, pause: int = PAUSE_SECONDES, max_retries: int = 3) -> BeautifulSoup | None:
    """Télécharge une page et retourne un objet BeautifulSoup. Retourne None en cas d'échec."""
    for tentative in range(1, max_retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            logger.info(f"GET {url} → HTTP {response.status_code}")

            if response.status_code == 200:
                time.sleep(pause)  # politesse : on attend avant la prochaine requête
                return BeautifulSoup(response.text, "lxml")

            elif response.status_code in (429, 503):
                attente = pause * (2 ** tentative)  # backoff exponentiel
                logger.warning(f"Code {response.status_code} — retry {tentative}/{max_retries} dans {attente}s")
                time.sleep(attente)

            else:
                logger.error(f"Code inattendu {response.status_code} pour {url}")
                return None

        except requests.exceptions.Timeout:
            logger.warning(f"Timeout sur {url} (tentative {tentative}/{max_retries})")
        except requests.exceptions.ConnectionError:
            logger.error(f"Erreur de connexion sur {url}")
            return None

    logger.error(f"Échec après {max_retries} tentatives pour {url}")
    return None

print("Fonction get_page() définie ✓")

Fonction get_page() définie ✓


---
## Étape 3 — Scraping (choisissez votre source)

Deux sources sont proposées. **Choisissez celle que vous avez retenue en Phase 1.**  
Déroulez uniquement la section correspondante.

---
### Option A — Wikipedia (page de catégorie)

Nous allons scraper la liste des articles dans une catégorie Wikipedia.  
La structure HTML est stable et bien documentée.

In [14]:
# ============================================================
# OPTION A — Wikipedia : catégorie d'articles
# Vous pouvez changer la catégorie ci-dessous
# ============================================================

CATEGORIE_WIKIPEDIA = "Intelligence_artificielle"  # ✏️ Modifiez si vous voulez une autre catégorie
NB_PAGES_MAX = 3       # ✏️ Nombre de pages de catégorie à parcourir (chaque page ≈ 200 articles)

BASE_URL  = "https://fr.wikipedia.org"
SOURCE    = "wikipedia.fr"

articles  = []
next_url  = f"{BASE_URL}/wiki/Cat%C3%A9gorie:{CATEGORIE_WIKIPEDIA}"
page_num  = 0

while next_url and page_num < NB_PAGES_MAX:
    page_num += 1
    print(f"\n--- Page {page_num} : {next_url}")

    soup = get_page(next_url)
    if soup is None:
        print("⚠️  Page inaccessible, on arrête.")
        break

    # Extraction des articles listés dans la catégorie
    # Les titres sont dans : div#mw-pages > div.mw-category > ul > li > a
    liens = soup.select("#mw-pages .mw-category-group li a")

    if not liens:
        # Essai avec un sélecteur alternatif (la structure peut varier selon les catégories)
        liens = soup.select("#mw-pages li a")

    for lien in liens:
        titre = lien.get_text(strip=True)
        href  = lien.get("href", "")
        url   = BASE_URL + href if href.startswith("/") else href

        if titre and url:
            articles.append({
                "source": SOURCE,
                "titre":  titre,
                "date":   "",      # Les pages de catégorie ne donnent pas de date directement
                "url":    url,
            })

    print(f"  → {len(liens)} articles trouvés (total : {len(articles)})")

    # Pagination : lien "page suivante"
    next_link = soup.find("a", string=lambda t: t and "page suivante" in t.lower())
    next_url  = BASE_URL + next_link["href"] if next_link else None

print(f"\n✅ Collecte terminée : {len(articles)} articles au total")


--- Page 1 : https://fr.wikipedia.org/wiki/Cat%C3%A9gorie:Intelligence_artificielle


13:29:32 | INFO | GET https://fr.wikipedia.org/wiki/Cat%C3%A9gorie:Intelligence_artificielle → HTTP 200


  → 200 articles trouvés (total : 200)

--- Page 2 : https://fr.wikipedia.org/w/index.php?title=Cat%C3%A9gorie:Intelligence_artificielle&pagefrom=Kizuna+AI#mw-pages


13:29:34 | INFO | GET https://fr.wikipedia.org/w/index.php?title=Cat%C3%A9gorie:Intelligence_artificielle&pagefrom=Kizuna+AI#mw-pages → HTTP 200


  → 173 articles trouvés (total : 373)

✅ Collecte terminée : 373 articles au total


In [15]:
# Récupération des dates de dernière modification (optionnel, bonus)
# On visite les 20 premiers articles pour enrichir la colonne 'date'
# ⚠️  Cela fait des requêtes supplémentaires — assurez-vous d'avoir le temps

ENRICHIR_DATES = False  # ✏️ Passez à True pour activer (prend ~1-2 min de plus)

if ENRICHIR_DATES:
    print("Récupération des dates (20 premiers articles)...")
    for i, article in enumerate(articles[:20]):
        soup = get_page(article["url"])
        if soup:
            # Cherche la balise <time> de dernière modification
            time_tag = soup.find("li", id="footer-info-lastmod")
            if time_tag:
                articles[i]["date"] = time_tag.get_text(strip=True).replace("Cette page a été modifiée pour la dernière fois le ", "")[:10]
        print(f"  {i+1}/20", end="\r")
    print("\nDates récupérées ✓")
else:
    print("Enrichissement des dates désactivé (ENRICHIR_DATES = False)")

Enrichissement des dates désactivé (ENRICHIR_DATES = False)


---
### Option B — data.gouv.fr (liste de datasets)

Si vous avez choisi data.gouv.fr, exécutez cette section à la place de l'Option A.

In [18]:
# ============================================================
# OPTION B — data.gouv.fr : liste de datasets via l'API
# L'API publique est explicitement autorisée (open data)
# ============================================================

# Décommentez ce bloc si vous avez choisi data.gouv.fr

SUJET         = "education"     # ✏️ Modifiez le thème de recherche
NB_PAGES_MAX  = 3
SOURCE        = "data.gouv.fr"
articles      = []

for page in range(1, NB_PAGES_MAX + 1):
    api_url = f"https://www.data.gouv.fr/api/1/datasets/?q={SUJET}&page={page}&page_size=20"
    print(f"Page {page} : {api_url}")

    try:
        r = requests.get(api_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
        logger.info(f"GET {api_url} → HTTP {r.status_code}")
        time.sleep(PAUSE_SECONDES)

        for dataset in data.get("data", []):
            articles.append({
                "source": SOURCE,
                "titre":  dataset.get("title", ""),
                "date":   dataset.get("last_modified", "")[:10] if dataset.get("last_modified") else "",
                "url":    f"https://www.data.gouv.fr/fr/datasets/{dataset.get('slug', '')}/",
            })

        if not data.get("data"):
            print("Plus de résultats.")
            break

    except Exception as e:
        print(f"Erreur page {page} : {e}")
        break

print(f"\n✅ Collecte terminée : {len(articles)} datasets")

print("Option B désactivée — décommentez ce bloc si vous avez choisi data.gouv.fr")

Page 1 : https://www.data.gouv.fr/api/1/datasets/?q=education&page=1&page_size=20


13:31:42 | INFO | GET https://www.data.gouv.fr/api/1/datasets/?q=education&page=1&page_size=20 → HTTP 200


Page 2 : https://www.data.gouv.fr/api/1/datasets/?q=education&page=2&page_size=20


13:31:45 | INFO | GET https://www.data.gouv.fr/api/1/datasets/?q=education&page=2&page_size=20 → HTTP 200


Page 3 : https://www.data.gouv.fr/api/1/datasets/?q=education&page=3&page_size=20
Erreur page 3 : 404 Client Error: NOT FOUND for url: https://www.data.gouv.fr/api/1/datasets/?q=education&page=3&page_size=20

✅ Collecte terminée : 30 datasets
Option B désactivée — décommentez ce bloc si vous avez choisi data.gouv.fr


---
## Étape 4 — Contrôle qualité rapide & export CSV

In [19]:
# Création du DataFrame
df = pd.DataFrame(articles, columns=["source", "titre", "date", "url"])

print(f"Nombre de lignes collectées : {len(df)}")
print(f"Colonnes : {list(df.columns)}")
print()
print("=== Aperçu (5 premières lignes) ===")
display(df.head())

print("\n=== Valeurs manquantes ===")
print(df.isna().sum())

print("\n=== Doublons ===")
print(f"Doublons sur (titre, url) : {df.duplicated(subset=['titre','url']).sum()}")

Nombre de lignes collectées : 30
Colonnes : ['source', 'titre', 'date', 'url']

=== Aperçu (5 premières lignes) ===


,source,titre,date,url
0,data.gouv.fr,Education,2019-03-19,https://www.data.gouv.fr/fr/datasets/education-1/
1,data.gouv.fr,Académies Education Prioritaire,2026-01-29,https://www.data.gouv.fr/fr/datasets/academies...
2,data.gouv.fr,OpenStreetMap : Education - points,2025-06-07,https://www.data.gouv.fr/fr/datasets/openstree...
3,data.gouv.fr,Séries chronologiques Education,2018-06-12,https://www.data.gouv.fr/fr/datasets/series-ch...
4,data.gouv.fr,OpenStreetMap : Education - polygones,2025-06-04,https://www.data.gouv.fr/fr/datasets/openstree...



=== Valeurs manquantes ===
source    0
titre     0
date      0
url       0
dtype: int64

=== Doublons ===
Doublons sur (titre, url) : 1


In [20]:
# Vérification du seuil minimal de 20 lignes
SEUIL_MIN = 20

if len(df) >= SEUIL_MIN:
    chemin_export = "data/export.csv"
    df.to_csv(chemin_export, index=False, encoding="utf-8")
    print(f"✅ Export réussi : {chemin_export} ({len(df)} lignes)")
else:
    print(f"⚠️  Seulement {len(df)} lignes collectées (minimum requis : {SEUIL_MIN}).")
    print("   Recommandation : augmentez NB_PAGES_MAX ou passez au dataset de secours.")
    if len(df) > 0:
        df.to_csv("data/export_partiel.csv", index=False, encoding="utf-8")
        print("   Export partiel sauvegardé dans data/export_partiel.csv")

✅ Export réussi : data/export.csv (30 lignes)


---
## Étape 5 — Justification de vos choix (à rédiger)

Complétez la cellule ci-dessous. Ces réponses font partie du livrable noté.

### Justification technique (à compléter)

**1. Source choisie et pourquoi :**  
*à compléter — ex: Wikipedia car autorisée par robots.txt, structure HTML stable, pas de JS dynamique*

**2. Sélecteur CSS / XPath utilisé :**  
*à compléter — ex: `#mw-pages .mw-category-group li a` car il cible précisément la liste d'articles*

**3. Pause appliquée et justification :**  
*à compléter — ex: 2 secondes car aucun crawl-delay défini dans robots.txt, 2s est une valeur prudente*

**4. Gestion des erreurs :**  
*à compléter — ex: retry avec backoff sur 429/503, timeout de 15s, champ vide si balise absente*

**5. Difficultés rencontrées (le cas échéant) :**  
*à compléter*

---
## ✅ Fin de la Phase 2

Avant de passer au notebook `03_audit.ipynb`, vérifiez :
- [ ] `data/export.csv` existe et contient ≥ 20 lignes
- [ ] Les 4 colonnes (`source`, `titre`, `date`, `url`) sont présentes
- [ ] Votre justification est rédigée

Si vous n'avez pas de CSV valide : utilisez `data/dataset_secours.csv` → **passez à `03_audit.ipynb`**